[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C07_ML_Foundations_Course/03_supervised_learning_scratch/03_supervised_learning_scratch.ipynb)

# 03 · 经典监督学习从零

<span style="background:#1f6feb;color:#fff;padding:2px 8px;border-radius:4px;font-size:12px">CPU</span>
在 **California Housing**（真实房价回归）和 **UCI 乳腺癌**（分类）上，从零写 OLS、逻辑回归、决策树、bagging。

**你将完成：**
1. OLS 闭式解 + R²（预测加州房价）
2. 逻辑回归梯度下降（诊断肿瘤）
3. Gini 最优切分 / 决策树桩
4. Bagging：一堆树桩平均，打败单棵树

> 数据：California Housing（20640 区块, 1990 普查）+ UCI WDBC。

## 0 · 加载真实数据

In [ ]:
import os, urllib.request
import numpy as np, pandas as pd
np.set_printoptions(precision=4, suppress=True)
CACHE=os.path.expanduser("~/.ml_foundations_data"); os.makedirs(CACHE, exist_ok=True)
def fetch(u,f):
    p=os.path.join(CACHE,f)
    if not os.path.exists(p): urllib.request.urlretrieve(u,p)
    return p

# 回归：加州房价
hou = pd.read_csv(fetch("https://raw.githubusercontent.com/ageron/handson-ml2/master/datasets/housing/housing.csv","housing.csv"))
hou = hou.dropna()
feat = ["median_income","housing_median_age","total_rooms","total_bedrooms","population","households"]
Xr = hou[feat].to_numpy(float); yr = (hou["median_house_value"].to_numpy(float))/1e5  # 单位:十万美元
print("房价回归 X:", Xr.shape, " y(十万$) 范围:", yr.min().round(2), "~", yr.max().round(2))

# 分类：乳腺癌
df = pd.read_csv(fetch("https://archive.ics.uci.edu/ml/machine-learning-databases/breast-cancer-wisconsin/wdbc.data","wdbc.data"), header=None)
Xc = df.iloc[:,2:].to_numpy(float); yc = (df[1].to_numpy()=="M").astype(float)
print("乳腺癌分类 X:", Xc.shape, " 恶性占比:", yc.mean().round(3))

def split_std(X, y, frac=0.7, seed=0, standardize=True):
    rng=np.random.default_rng(seed); idx=rng.permutation(len(y)); cut=int(frac*len(idx))
    tr,te=idx[:cut],idx[cut:]
    if standardize:
        mu,sd=X[tr].mean(0),X[tr].std(0); Xtr,Xte=(X[tr]-mu)/sd,(X[te]-mu)/sd
    else:
        Xtr,Xte=X[tr],X[te]
    return Xtr,Xte,y[tr],y[te]

## 1 · OLS 闭式解 + R²（加州房价）

$\hat w=(X^\top X)^{-1}X^\top y$，用 `lstsq` 更稳。加一列 1 当截距。R² 衡量解释了多少方差。

In [ ]:
Xtr,Xte,ytr,yte = split_std(Xr, yr, standardize=True)
def add_bias(X): return np.c_[X, np.ones(len(X))]
w,_,_,_ = np.linalg.lstsq(add_bias(Xtr), ytr, rcond=None)

def r2(y, yhat): return 1 - ((y-yhat)**2).sum()/((y-y.mean())**2).sum()
pred = add_bias(Xte) @ w
print(f"test R² = {r2(yte, pred):.3f}  (1=完美, 0=只会猜均值)")
print("最重要特征（标准化后系数绝对值最大）:", feat[np.argmax(np.abs(w[:-1]))],
      "=> median_income 主导房价，符合直觉")

## 2 · 逻辑回归梯度下降（乳腺癌）

无闭式解，用 GD。梯度 = `Xᵀ(σ(Xw)-y)/n`。

In [ ]:
Xtr,Xte,ytr,yte = split_std(Xc, yc, standardize=True)
def sigmoid(z): return 1/(1+np.exp(-np.clip(z,-500,500)))
w = np.zeros(Xtr.shape[1]); b = 0.0; lr = 0.5
for _ in range(500):
    p = sigmoid(Xtr@w + b); g = p - ytr
    w -= lr * Xtr.T@g/len(ytr); b -= lr * g.mean()
acc = ((sigmoid(Xte@w+b)>0.5)==yte).mean()
print(f"逻辑回归 test acc = {acc:.3f}  (baseline={max(yte.mean(),1-yte.mean()):.3f})")

## 3 · Gini 最优切分（决策树桩）

枚举一个特征上的所有阈值，找加权 Gini 最小的切分。这是决策树的原子操作。

In [ ]:
def gini(y):
    if len(y)==0: return 0.0
    p = np.bincount(y.astype(int), minlength=2)/len(y)
    return 1 - (p**2).sum()

def best_split_feature(x, y):
    order=np.argsort(x); xs,ys=x[order],y[order]; n=len(y)
    best=(None, float("inf"))
    for i in range(1,n):
        if xs[i]==xs[i-1]: continue
        thr=(xs[i]+xs[i-1])/2
        left,right=ys[:i],ys[i:]
        wg=(len(left)*gini(left)+len(right)*gini(right))/n
        if wg<best[1]: best=(thr,wg)
    return best

# 在乳腺癌某个特征上找最优切分
j=np.argmax([abs(np.corrcoef(Xtr[:,k],ytr)[0,1]) for k in range(Xtr.shape[1])])
thr,imp=best_split_feature(Xtr[:,j], ytr)
print(f"最相关特征 #{j}  最优阈值={thr:.3f}  加权Gini={imp:.4f}")
stump_acc=(((Xte[:,j]>thr).astype(float)==yte).mean())
stump_acc=max(stump_acc,1-stump_acc)
print(f"单树桩 test acc ≈ {stump_acc:.3f}")

## 4 · Bagging：树桩的力量在数量

单个树桩很弱。但在 bootstrap 子样本上训练很多个、再投票，方差大幅下降。

In [ ]:
def fit_stump(X, y):
    best=(None,None,float("inf"))
    for j in range(X.shape[1]):
        thr,imp=best_split_feature(X[:,j], y)
        if thr is not None and imp<best[2]: best=(j,thr,imp)
    j,thr,_=best
    # 决定哪边是 1
    left_label = round(y[X[:,j]<=thr].mean()) if (X[:,j]<=thr).any() else 0
    return (j,thr,left_label)
def pred_stump(s, X):
    j,thr,ll=s; left=X[:,j]<=thr
    return np.where(left, ll, 1-ll).astype(float)

def bagging_acc(seed, B=51):
    rng=np.random.default_rng(seed); stumps=[]
    for _ in range(B):
        idx=rng.integers(0,len(ytr),len(ytr))  # bootstrap
        stumps.append(fit_stump(Xtr[idx], ytr[idx]))
    votes=np.mean([pred_stump(s,Xte) for s in stumps],axis=0)
    return ((votes>0.5)==yte).mean()

# 单次 bagging 有随机性（单棵树桩偶尔会碰巧打平某一次），bagging 的本质是"降方差"——
# 在多个随机种子上取平均，才能稳健看到它 >= 单树桩。
B=51; seeds=range(10)
bag_accs=[bagging_acc(s, B) for s in seeds]
bag_acc=float(np.mean(bag_accs))
print(f"单树桩 acc≈{stump_acc:.3f}  ->  {B}个树桩 bagging acc(在 {len(list(seeds))} 个种子上平均)={bag_acc:.3f}")
print(f"   （单次 bagging 波动范围 [{min(bag_accs):.3f}, {max(bag_accs):.3f}]）")
print("=> bagging 通过降方差稳健地提升了弱学习器")

## 5 · Boosting：串行拟合残差降偏差（对比 bagging 降方差）

§4 的 bagging 通过平均许多高方差模型来**降方差**。Boosting 是另一条路：串行地让每个新的弱学习器去拟合**前面所有模型的残差**（平方损失下残差正是负梯度，这就是 gradient boosting），从而把弱学习器的**偏差**一步步压下来。下面在真实加州房价回归上，用一个故意很弱（高偏差、欠拟合）的「回归树桩」当基学习器，看 boosting 50 轮如何把 test R² 从单桩的低值一路抬上来——这正是 GBDT / XGBoost 的核心机制。

In [ ]:
# Boosting：串行拟合「上一轮的残差」，降偏差。在真实加州房价回归上演示。
# 弱学习器用「单层回归树桩」（按一个特征的一个阈值分两段、各段取均值），故意很弱（高偏差）。
Xtr_r, Xte_r, ytr_r, yte_r = split_std(Xr, yr, standardize=True)

def fit_reg_stump(X, r):
    # 找使残差平方和(SSE)最小的 (特征, 阈值)；两段各预测段内均值
    best = (0, 0.0, np.inf, r.mean(), r.mean())
    for j in range(X.shape[1]):
        xs = X[:, j]; order = np.argsort(xs); xss = xs[order]; rss = r[order]
        for q in np.linspace(0.1, 0.9, 9):                 # 候选分位阈值
            thr = np.quantile(xss, q)
            L = xs <= thr; R = ~L
            if L.sum() < 5 or R.sum() < 5: continue
            lv, rv = r[L].mean(), r[R].mean()
            sse = ((r[L]-lv)**2).sum() + ((r[R]-rv)**2).sum()
            if sse < best[2]: best = (j, thr, sse, lv, rv)
    return best
def pred_reg_stump(s, X):
    j, thr, _, lv, rv = s
    return np.where(X[:, j] <= thr, lv, rv)

def r2(y, yhat): return 1 - ((y-yhat)**2).sum()/((y-y.mean())**2).sum()

# 单个弱树桩 baseline
s0 = fit_reg_stump(Xtr_r, ytr_r)
r2_single = r2(yte_r, pred_reg_stump(s0, Xte_r))

# Boosting：F <- F + lr * stump(残差)
F_tr = np.full_like(ytr_r, ytr_r.mean()); F_te = np.full_like(yte_r, ytr_r.mean())
lr = 0.3; M = 50; trace = []
for m in range(M):
    resid = ytr_r - F_tr                      # 当前残差 = 负梯度（平方损失）
    s = fit_reg_stump(Xtr_r, resid)
    F_tr = F_tr + lr * pred_reg_stump(s, Xtr_r)
    F_te = F_te + lr * pred_reg_stump(s, Xte_r)
    if m in (0, 4, 19, 49): trace.append((m+1, r2(yte_r, F_te)))

print(f"单个弱回归树桩  test R² = {r2_single:.3f}  (高偏差、欠拟合)")
for m, sc in trace: print(f"  boosting {m:2d} 轮后  test R² = {sc:.3f}")
print("=> boosting 串行拟合残差，把弱学习器的偏差一步步压下来（bias↓）；对比 §4 bagging 是降方差（variance↓）")

assert trace[-1][1] > r2_single + 0.1, "boosting 多轮后应显著优于单个弱树桩（降偏差生效）"
print("boosting 验证通过 ✓")

---
## ✏️ 练习区

### ✏️ 练习 1：OLS 闭式解 + R²

实现 `fit_ols(X,y)` 返回带截距的 `(w, b)`，和 `r2_score(y, yhat)`。

In [ ]:
def fit_ols(X, y):
    # TODO: 拼一列1，用 np.linalg.lstsq；返回 (权重向量不含截距, 截距)
    raise NotImplementedError
def r2_score(y, yhat):
    # TODO: 1 - SS_res/SS_tot
    raise NotImplementedError


In [ ]:
# —— 练习 1 自测（真实房价）——
Xtr,Xte,ytr,yte = split_std(Xr, yr, standardize=True)
w,b = fit_ols(Xtr, ytr)
assert w.shape==(Xtr.shape[1],)
pred = Xte@w + b
score = r2_score(yte, pred)
assert score > 0.5, "加州房价标准化后 R² 应 >0.5"
assert abs(r2_score(yte, np.full_like(yte, ytr.mean())) - 0) < 0.05, "猜均值 R²≈0"
print(f"练习 1 通过 ✓  test R²={score:.3f}")


### ✏️ 练习 2：逻辑回归（带 L2 正则）

实现 `fit_logreg(X,y,lr,steps,l2)`，返回 `(w,b)`。梯度里加 L2 项 `l2*w`。

In [ ]:
def fit_logreg(X, y, lr=0.5, steps=500, l2=0.0):
    # TODO: GD；梯度 = X.T@(p-y)/n + l2*w
    raise NotImplementedError


In [ ]:
# —— 练习 2 自测（乳腺癌）——
Xtr,Xte,ytr,yte = split_std(Xc, yc, standardize=True)
w,b = fit_logreg(Xtr, ytr, lr=0.5, steps=500, l2=0.01)
acc = ((1/(1+np.exp(-(Xte@w+b)))>0.5)==yte).mean()
assert acc > 0.93, "乳腺癌标准化后应轻松 >93%"
print(f"练习 2 通过 ✓  test acc={acc:.3f}")


### ✏️ 练习 3：information gain

实现 `info_gain(parent_y, left_y, right_y)`：用 Gini 算切分前后的纯度提升（加权）。

In [ ]:
def info_gain(parent_y, left_y, right_y):
    # TODO: gini(parent) - 加权平均(gini(left), gini(right))
    raise NotImplementedError


In [ ]:
# —— 练习 3 自测 ——
parent=np.array([0,0,1,1]);
assert abs(info_gain(parent, np.array([0,0]), np.array([1,1])) - 0.5) < 1e-9, "完美切分 gain=0.5"
assert abs(info_gain(parent, np.array([0,1]), np.array([0,1])) - 0.0) < 1e-9, "没用的切分 gain=0"
print("练习 3 通过 ✓")


### ✏️ 练习 4：bagging 集成

实现 `bagging_predict(Xtr,ytr,Xte,B,seed)`：训练 B 个 bootstrap 树桩，返回测试集多数投票预测。
bagging 靠**降方差**取胜——单次结果有随机性，要在多个种子上取**平均准确率**才能稳健体现 bagging ≥ 单树桩。

In [ ]:
def bagging_predict(Xtr, ytr, Xte, B=51, seed=0):
    # TODO: 用提供的 fit_stump/pred_stump，bootstrap B 次，平均投票>0.5
    raise NotImplementedError


In [ ]:
# —— 练习 4 自测 ——
# 单次 bagging 受随机种子影响有方差，单棵决策树桩偶尔会碰巧打平甚至略胜某一次 bagging。
# bagging 的本质是"降方差"，要在多个种子上取平均才能稳健地体现它 >= 单模型。
single = fit_stump(Xtr, ytr); s_acc = (pred_stump(single, Xte) == yte).mean()
bag_accs = [(bagging_predict(Xtr, ytr, Xte, B=51, seed=s) == yte).mean() for s in range(10)]
bag = float(np.mean(bag_accs))
assert bag >= s_acc - 1e-9, "多个种子上平均，bagging 不应比单树桩差（降方差）"
print(f"练习 4 通过 ✓  单树桩 {s_acc:.3f} -> bagging(10 个种子平均) {bag:.3f}")


---
## 📖 参考答案

In [ ]:
# 练习 1
def fit_ols(X, y):
    A=np.c_[X, np.ones(len(X))]; theta,*_=np.linalg.lstsq(A,y,rcond=None)
    return theta[:-1], float(theta[-1])
def r2_score(y, yhat):
    return 1 - ((y-yhat)**2).sum()/((y-y.mean())**2).sum()
print("练习 1 ✓")

In [ ]:
# 练习 2
def fit_logreg(X, y, lr=0.5, steps=500, l2=0.0):
    w=np.zeros(X.shape[1]); b=0.0
    for _ in range(steps):
        p=1/(1+np.exp(-np.clip(X@w+b,-500,500))); g=p-y
        w-=lr*(X.T@g/len(y)+l2*w); b-=lr*g.mean()
    return w,b
print("练习 2 ✓")

In [ ]:
# 练习 3
def info_gain(parent_y, left_y, right_y):
    n=len(parent_y)
    return gini(parent_y) - (len(left_y)*gini(left_y)+len(right_y)*gini(right_y))/n
print("练习 3 ✓")

In [ ]:
# 练习 4
def bagging_predict(Xtr, ytr, Xte, B=51, seed=0):
    rng=np.random.default_rng(seed); preds=[]
    for _ in range(B):
        idx=rng.integers(0,len(ytr),len(ytr))
        preds.append(pred_stump(fit_stump(Xtr[idx],ytr[idx]), Xte))
    return (np.mean(preds,axis=0)>0.5).astype(float)
print("练习 4 ✓ —— 这就是随机森林的核心思想")